In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
from argparse import Namespace
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# project imports
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from ml.utils.data_utils import prepare_dataset
from ml.models.lstm import LSTM
from ml.models.multi_step_lstm import MultiStepLSTM
from ml.models.seq2seq_lstm import Seq2SeqLSTM
from ml.models.transformer import TimeSeriesTransformer

In [ ]:
# -----------------------------
# 0. CONFIG
# -----------------------------
DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TARGETS = ['rnti_count', 'rb_down', 'rb_up', 'down', 'up']
H       = 6  # forecast horizon

# DATA PATHS
FULL_BASE_PATH      = '../dataset/full_dataset.csv'
FULL_EXTRA_PATH     = '../dataset/full_dataset_with_extra_data_26Aug.csv'  # (no extra-trained full-model ckpts provided)
CLU_BASE_PATH       = '../dataset/combined_with_cluster_feature.csv'
CLU_EXTRA_PATH      = '../dataset/combined_with_cluster_feature_with_extraData_26Aug.csv'

# CHECKPOINTS (trained on BASE data)
CKPT_BASE_T1        = "base_lstm_t1.pt"
CKPT_MULTI          = "multi_step_lstm.pt"
CKPT_S2S_MULTI      = "seq2seq_lstm_multistep.pth"
CKPT_TRANS          = "transformer_multistep.pt"
CKPT_S2S_CLU        = "seq2seq_cluster_huber.pt"
CKPT_TRANS_CLU      = "transformer_multistep_cluster.pt"

# CHECKPOINTS (trained on BASE + EXTRA data)
CKPT_S2S_CLU_EXTRA  = "seq2seq_cluster_with_extra_data_26Aug.pt"
CKPT_TRANS_CLU_EXTRA= "transformer_multistep_cluster_with_extra_data_26Aug.pt"

In [ ]:
# -----------------------------
# 1. HELPERS
# -----------------------------
def metrics_scaled(y_true_2d, y_pred_2d):
    mse  = mean_squared_error(y_true_2d, y_pred_2d)
    rmse = mean_squared_error(y_true_2d, y_pred_2d, squared=False)
    mae  = mean_absolute_error(y_true_2d, y_pred_2d)
    r2   = r2_score(y_true_2d, y_pred_2d)
    nrmse = rmse / (np.max(y_true_2d) - np.min(y_true_2d) + 1e-8)
    return {"MSE": mse, "RMSE": rmse, "MAE": mae, "R2": r2, "NRMSE": nrmse}

def mape(y_true, y_pred, eps=1e-8):
    denom = np.clip(np.abs(y_true), eps, None)
    return float(np.mean(np.abs((y_true - y_pred)/denom))*100.0)

def smape(y_true, y_pred, eps=1e-8):
    denom = np.clip((np.abs(y_true)+np.abs(y_pred))/2.0, eps, None)
    return float(np.mean(np.abs(y_true - y_pred)/denom)*100.0)

def inverse_single_col(y_scaled_1d, scaler, j):
    # MinMax or Standard
    if hasattr(scaler, "min_") and hasattr(scaler, "scale_"):
        return (np.asarray(y_scaled_1d) - scaler.min_[j]) / scaler.scale_[j]
    if hasattr(scaler, "mean_") and hasattr(scaler, "scale_"):
        return np.asarray(y_scaled_1d) * scaler.scale_[j] + scaler.mean_[j]
    raise ValueError("Unknown scaler type for inverse transform.")

def percent_metrics_original_col(y_true_scaled_1d, y_pred_scaled_1d, scaler, j):
    yt = inverse_single_col(y_true_scaled_1d, scaler, j)
    yp = inverse_single_col(y_pred_scaled_1d, scaler, j)
    return {"MAPE%": mape(yt, yp), "sMAPE%": smape(yt, yp)}

def roll_base_lstm_to_horizon(base_model, X_init, steps, target_pos_in_x, device="cpu"):
    base_model.eval()
    x_win = torch.tensor(X_init, dtype=torch.float32, device=device)
    outs = []
    with torch.no_grad():
        for _ in range(steps):
            y_next = base_model(x_win, device=device)  # [N,T]
            outs.append(y_next.unsqueeze(1))
            last_row = x_win[:, -1, :].clone()
            for k, pos in enumerate(target_pos_in_x):
                last_row[:, pos] = y_next[:, k]
            x_win = torch.cat([x_win[:, 1:, :], last_row.unsqueeze(1)], dim=1)
    return torch.cat(outs, dim=1).cpu().numpy()

def infer_target_positions_from_data(X_test, y_t1_scaled):
    N, L, D = X_test.shape
    T = y_t1_scaled.shape[1]
    X_last = X_test[:, -1, :]
    pos, used = [], set()
    for i in range(T):
        yt = y_t1_scaled[:, i]
        yt = yt - yt.mean(); yt_std = yt.std() + 1e-12
        corrs = []
        for j in range(D):
            xj = X_last[:, j]; xj = xj - xj.mean()
            xj_std = xj.std() + 1e-12
            corr = float(np.mean((xj/xj_std)*(yt/yt_std)))
            corrs.append(abs(corr))
        for j in np.argsort(corrs)[::-1]:
            if j not in used:
                pos.append(int(j)); used.add(int(j)); break
    return pos if len(pos)==T else list(range(T))

def eval_strategy_A_block(name, y_pred_scaled, y_true_scaled, scaler):
    rows = []
    for j, tgt in enumerate(TARGETS):
        m_core = metrics_scaled(y_true_scaled[:, j:j+1], y_pred_scaled[:, j:j+1])
        m_pct  = percent_metrics_original_col(y_true_scaled[:, j], y_pred_scaled[:, j], scaler, j)
        rows.append({
            "Strategy": "A_t+1", "Model": name, "Target": tgt,
            **m_core, **m_pct
        })
    return rows

def eval_strategy_B_block(name, y_pred_scaled, y_true_scaled, scaler, tag="B"):
    rows_steps, rows_over = [], []
    # per-step
    for step in range(y_true_scaled.shape[1]):
        for j, tgt in enumerate(TARGETS):
            yt = y_true_scaled[:, step, j]; yp = y_pred_scaled[:, step, j]
            m_core = metrics_scaled(yt.reshape(-1,1), yp.reshape(-1,1))
            m_pct  = percent_metrics_original_col(yt, yp, scaler, j)
            rows_steps.append({
                "Strategy": f"{tag}_t+{step+1}", "Step": step+1, "Model": name,
                "Target": tgt, **m_core, **m_pct
            })
    # overall
    for j, tgt in enumerate(TARGETS):
        yt_all = y_true_scaled[:, :, j].reshape(-1)
        yp_all = y_pred_scaled[:, :, j].reshape(-1)
        m_core = metrics_scaled(yt_all.reshape(-1,1), yp_all.reshape(-1,1))
        m_pct  = percent_metrics_original_col(yt_all, yp_all, scaler, j)
        rows_over.append({
            "Strategy": f"{tag}_overall", "Model": name, "Target": tgt, **m_core, **m_pct
        })
    return rows_steps, rows_over

def load_split(data_path, use_time_features=False):
    args = Namespace(
        data_path=data_path, targets=TARGETS, num_lags=10, forecast_steps=H,
        test_size=0.2, ignore_cols=None, identifier='District', nan_constant=0,
        x_scaler='minmax', y_scaler='minmax', outlier_detection=True,
        batch_size=128, cuda=torch.cuda.is_available(), seed=42,
        use_time_features=use_time_features
    )
    return prepare_dataset(args)

In [ ]:
# -----------------------------
# 2. FAMILY 1: FULL (no cluster) – BASE DATA ONLY
# -----------------------------
print("\n=== FAMILY 1: FULL (no cluster) – BASE DATA ONLY ===")
X_tr, y_tr, X_te, y_te, xsc_f, ysc_f, *_ = load_split(FULL_BASE_PATH)
N, L, D = X_te.shape; T = y_te.shape[2]
y_te_t1 = y_te[:, 0, :]  # scaled

# models
base_m = LSTM(input_dim=D, lstm_hidden_size=128, num_lstm_layers=2,
              lstm_dropout=0.0, layer_units=[128,64], num_outputs=T,
              matrix_rep=True, exogenous_dim=0).to(DEVICE)
base_m.load_state_dict(torch.load(CKPT_BASE_T1, map_location=DEVICE), strict=True)
base_m.eval()

basic_m = MultiStepLSTM(input_size=D, hidden_size=128, num_layers=1,
                        output_size=T, forecast_steps=H).to(DEVICE)
basic_m.load_state_dict(torch.load(CKPT_MULTI, map_location=DEVICE), strict=True)
basic_m.eval()

s2s_m = Seq2SeqLSTM(input_size=D, hidden_size=64, output_size=T,
                    forecast_steps=H, num_layers=1).to(DEVICE)
s2s_m.load_state_dict(torch.load(CKPT_S2S_MULTI, map_location=DEVICE), strict=True)
s2s_m.eval()

trans_m = TimeSeriesTransformer(input_size=D, output_size=T, forecast_steps=H,
                                d_model=128, nhead=4, num_encoder_layers=2,
                                num_decoder_layers=2, dim_feedforward=256,
                                dropout=0.1).to(DEVICE)
trans_m.load_state_dict(torch.load(CKPT_TRANS, map_location=DEVICE), strict=True)
trans_m.eval()

with torch.no_grad():
    xb = torch.tensor(X_te, dtype=torch.float32, device=DEVICE)
    base_t1_s  = base_m(xb, device=DEVICE).cpu().numpy()
    basic_all  = basic_m(xb).cpu().numpy()
    s2s_all    = s2s_m(xb, teacher_forcing_ratio=0.0).cpu().numpy()
    trans_all  = trans_m(xb).cpu().numpy()

# Strategy A – t+1
rowsA = []
rowsA += eval_strategy_A_block("Base LSTM (t+1)",          base_t1_s, y_te_t1, ysc_f)
rowsA += eval_strategy_A_block("Basic Multistep LSTM (t+1)", basic_all[:,0,:], y_te_t1, ysc_f)
rowsA += eval_strategy_A_block("Seq2Seq LSTM (t+1)",         s2s_all[:,0,:], y_te_t1, ysc_f)
rowsA += eval_strategy_A_block("Transformer (t+1)",          trans_all[:,0,:], y_te_t1, ysc_f)
df_A_full_base = pd.DataFrame(rowsA)

# Strategy B – t+1..t+6
def infer_pos(X, y_t1): return infer_target_positions_from_data(X, y_t1)
TARGET_POS_IN_X = infer_pos(X_te, y_te_t1)
base_roll = roll_base_lstm_to_horizon(base_m, X_te, H, TARGET_POS_IN_X, device=DEVICE)

rowsB_steps, rowsB_over = [], []
s, o = eval_strategy_B_block("Base LSTM (rolled)", base_roll, y_te, ysc_f, tag="B(FULL-BASE)")
rowsB_steps += s; rowsB_over += o
s, o = eval_strategy_B_block("Basic Multistep LSTM", basic_all, y_te, ysc_f, tag="B(FULL-BASE)")
rowsB_steps += s; rowsB_over += o
s, o = eval_strategy_B_block("Seq2Seq LSTM", s2s_all, y_te, ysc_f, tag="B(FULL-BASE)")
rowsB_steps += s; rowsB_over += o
s, o = eval_strategy_B_block("Transformer", trans_all, y_te, ysc_f, tag="B(FULL-BASE)")
rowsB_steps += s; rowsB_over += o
df_B_steps_full_base   = pd.DataFrame(rowsB_steps)
df_B_overall_full_base = pd.DataFrame(rowsB_over)


=== FAMILY 1: FULL (no cluster) – BASE DATA ONLY ===


In [ ]:
# -----------------------------
# 3. FAMILY 2: CLUSTERED – BASE DATA ONLY
# -----------------------------
print("\n=== FAMILY 2: CLUSTERED – BASE DATA ONLY ===")
X_tr_c, y_tr_c, X_te_c, y_te_c, xsc_c, ysc_c, *_ = load_split(CLU_BASE_PATH)
Nc, Lc, Dc = X_te_c.shape
y_te_c_t1 = y_te_c[:, 0, :]

s2s_c = Seq2SeqLSTM(input_size=Dc, hidden_size=64, output_size=T, forecast_steps=H, num_layers=1).to(DEVICE)
s2s_c.load_state_dict(torch.load(CKPT_S2S_CLU, map_location=DEVICE), strict=True)
s2s_c.eval()

trans_c = TimeSeriesTransformer(input_size=Dc, output_size=T, forecast_steps=H,
                                d_model=128, nhead=4, num_encoder_layers=2,
                                num_decoder_layers=2, dim_feedforward=256,
                                dropout=0.1).to(DEVICE)
trans_c.load_state_dict(torch.load(CKPT_TRANS_CLU, map_location=DEVICE), strict=True)
trans_c.eval()

with torch.no_grad():
    xb = torch.tensor(X_te_c, dtype=torch.float32, device=DEVICE)
    s2s_c_all   = s2s_c(xb, teacher_forcing_ratio=0.0).cpu().numpy()
    trans_c_all = trans_c(xb).cpu().numpy()

rowsA = []
rowsA += eval_strategy_A_block("Seq2Seq LSTM + Clusters (t+1)", s2s_c_all[:,0,:], y_te_c_t1, ysc_c)
rowsA += eval_strategy_A_block("Transformer + Clusters (t+1)",  trans_c_all[:,0,:], y_te_c_t1, ysc_c)
df_A_cluster_base = pd.DataFrame(rowsA)

rowsB_steps, rowsB_over = [], []
s, o = eval_strategy_B_block("Seq2Seq LSTM + Clusters", s2s_c_all, y_te_c, ysc_c, tag="B(CLU-BASE)")
rowsB_steps += s; rowsB_over += o
s, o = eval_strategy_B_block("Transformer + Clusters", trans_c_all, y_te_c, ysc_c, tag="B(CLU-BASE)")
rowsB_steps += s; rowsB_over += o
df_B_steps_cluster_base   = pd.DataFrame(rowsB_steps)
df_B_overall_cluster_base = pd.DataFrame(rowsB_over)


=== FAMILY 2: CLUSTERED – BASE DATA ONLY ===


In [ ]:
# -----------------------------
# 4. FAMILY 3: CLUSTERED – BASE + EXTRA DATA
# -----------------------------
print("\n=== FAMILY 3: CLUSTERED – BASE + EXTRA DATA ===")
X_tr_ce, y_tr_ce, X_te_ce, y_te_ce, xsc_ce, ysc_ce, *_ = load_split(CLU_EXTRA_PATH)
Ne, Le, De = X_te_ce.shape
y_te_ce_t1 = y_te_ce[:, 0, :]

s2s_ce = Seq2SeqLSTM(input_size=De, hidden_size=64, output_size=T, forecast_steps=H, num_layers=1).to(DEVICE)
s2s_ce.load_state_dict(torch.load(CKPT_S2S_CLU_EXTRA, map_location=DEVICE), strict=True)
s2s_ce.eval()

trans_ce = TimeSeriesTransformer(input_size=De, output_size=T, forecast_steps=H,
                                 d_model=128, nhead=4, num_encoder_layers=2,
                                 num_decoder_layers=2, dim_feedforward=256,
                                 dropout=0.1).to(DEVICE)
trans_ce.load_state_dict(torch.load(CKPT_TRANS_CLU_EXTRA, map_location=DEVICE), strict=True)
trans_ce.eval()

with torch.no_grad():
    xb = torch.tensor(X_te_ce, dtype=torch.float32, device=DEVICE)
    s2s_ce_all   = s2s_ce(xb, teacher_forcing_ratio=0.0).cpu().numpy()
    trans_ce_all = trans_ce(xb).cpu().numpy()

rowsA = []
rowsA += eval_strategy_A_block("Seq2Seq LSTM + Clusters + Extra (t+1)", s2s_ce_all[:,0,:], y_te_ce_t1, ysc_ce)
rowsA += eval_strategy_A_block("Transformer + Clusters + Extra (t+1)",  trans_ce_all[:,0,:], y_te_ce_t1, ysc_ce)
df_A_cluster_extra = pd.DataFrame(rowsA)

rowsB_steps, rowsB_over = [], []
s, o = eval_strategy_B_block("Seq2Seq LSTM + Clusters + Extra", s2s_ce_all, y_te_ce, ysc_ce, tag="B(CLU-EXTRA)")
rowsB_steps += s; rowsB_over += o
s, o = eval_strategy_B_block("Transformer + Clusters + Extra", trans_ce_all, y_te_ce, ysc_ce, tag="B(CLU-EXTRA)")
rowsB_steps += s; rowsB_over += o
df_B_steps_cluster_extra   = pd.DataFrame(rowsB_steps)
df_B_overall_cluster_extra = pd.DataFrame(rowsB_over)


=== FAMILY 3: CLUSTERED – BASE + EXTRA DATA ===


In [ ]:
# -----------------------------
# 5. PRINT OUTPUTS
# -----------------------------
print("\n=== Strategy A (t+1) – FULL (BASE ONLY) ===")
print(df_A_full_base.head(20).to_string(index=False))

print("\n=== Strategy B (overall) – FULL (BASE ONLY) ===")
print(df_B_overall_full_base.head(20).to_string(index=False))

print("\n=== Strategy A (t+1) – CLUSTER (BASE ONLY) ===")
print(df_A_cluster_base.head(20).to_string(index=False))

print("\n=== Strategy B (overall) – CLUSTER (BASE ONLY) ===")
print(df_B_overall_cluster_base.head(20).to_string(index=False))

print("\n=== Strategy A (t+1) – CLUSTER (+EXTRA) ===")
print(df_A_cluster_extra.head(20).to_string(index=False))

print("\n=== Strategy B (overall) – CLUSTER (+EXTRA) ===")
print(df_B_overall_cluster_extra.head(20).to_string(index=False))


=== Strategy A (t+1) – FULL (BASE ONLY) ===
Strategy                      Model     Target      MSE     RMSE      MAE       R2    NRMSE        MAPE%     sMAPE%
   A_t+1            Base LSTM (t+1) rnti_count 0.008045 0.089696 0.066063 0.485215 0.125351 2.548788e+01  24.742963
   A_t+1            Base LSTM (t+1)    rb_down 0.008326 0.091249 0.043579 0.559291 0.091397 4.725889e+01  36.282234
   A_t+1            Base LSTM (t+1)      rb_up 0.012164 0.110291 0.052190 0.610206 0.110291 2.719004e+03 123.955360
   A_t+1            Base LSTM (t+1)       down 0.007340 0.085676 0.050102 0.480701 0.085995 3.716739e+01  31.237519
   A_t+1            Base LSTM (t+1)         up 0.010708 0.103480 0.045972 0.556588 0.103480 2.214842e+13  95.755641
   A_t+1 Basic Multistep LSTM (t+1) rnti_count 0.006707 0.081893 0.062210 0.570880 0.114447 2.713494e+01  23.737117
   A_t+1 Basic Multistep LSTM (t+1)    rb_down 0.008512 0.092262 0.047042 0.549452 0.092412 5.133693e+01  36.843247
   A_t+1 Basic Multistep LS

In [19]:
# -----------------------------
# 6. SAVE ALL Tables
# -----------------------------
out_dir = '../dataset/capstone_results/'
os.makedirs(out_dir, exist_ok=True)
df_A_full_base.to_csv(os.path.join(out_dir, "A_full_base.csv"), index=False)
df_B_steps_full_base.to_csv(os.path.join(out_dir, "B_steps_full_base.csv"), index=False)
df_B_overall_full_base.to_csv(os.path.join(out_dir, "B_overall_full_base.csv"), index=False)

df_A_cluster_base.to_csv(os.path.join(out_dir, "A_cluster_base.csv"), index=False)
df_B_steps_cluster_base.to_csv(os.path.join(out_dir, "B_steps_cluster_base.csv"), index=False)
df_B_overall_cluster_base.to_csv(os.path.join(out_dir, "B_overall_cluster_base.csv"), index=False)

df_A_cluster_extra.to_csv(os.path.join(out_dir, "A_cluster_extra.csv"), index=False)
df_B_steps_cluster_extra.to_csv(os.path.join(out_dir, "B_steps_cluster_extra.csv"), index=False)
df_B_overall_cluster_extra.to_csv(os.path.join(out_dir, "B_overall_cluster_extra.csv"), index=False)